# XLSForm to Codebook Generator
This notebook parses an XLSForm survey and generates a readable Markdown codebook.

In [ ]:
import pandas as pd

# Load survey and choices
survey_path = '../input/survey.xlsx'
survey_df = pd.read_excel(survey_path, sheet_name='survey')
choices_df = pd.read_excel(survey_path, sheet_name='choices')

# Preview
survey_df.head()

In [ ]:
# Create choices lookup
choices_dict = choices_df.groupby('list_name').apply(
    lambda df: dict(zip(df['name'], df['label']))
).to_dict()

choices_dict

In [ ]:
# Process survey to extract codebook
codebook_lines = ["# Generated Codebook\n"]

for _, row in survey_df.iterrows():
    if pd.isna(row['name']) or row['type'].startswith("begin_group") or row['type'].startswith("end_group"):
        continue
    line = f"## `{row['name']}`\n"
    line += f"**Label:** {row['label']}\n\n"
    line += f"**Type:** {row['type']}\n\n"
    if pd.notna(row.get('hint')):
        line += f"**Hint:** {row['hint']}\n\n"
    if pd.notna(row.get('constraint')):
        line += f"**Constraint:** `{row['constraint']}`\n\n"
    if pd.notna(row.get('relevant')):
        line += f"**Relevant if:** `{row['relevant']}`\n\n"
    if "select_one" in row['type'] or "select_multiple" in row['type']:
        list_name = row['type'].split()[-1]
        options = choices_dict.get(list_name, {})
        line += "**Choices:**\n" + "\n".join([f"- `{k}`: {v}" for k, v in options.items()]) + "\n"
    codebook_lines.append(line)

# Save Markdown codebook
with open('../output/generated_codebook.md', 'w') as f:
    f.write("\n\n".join(codebook_lines))

Run all cells to generate `generated_codebook.md` in the `/output` folder.